# Test new intervention logic
* Compare previous strategy with strategy that compensates for the reconstruction error as per KD's suggestion

In [ ]:
!git clone https://github.com/mGarbowski/zzsn-projekt.git
!cd zzsn-projekt && git checkout better-intervention

In [ ]:
!wandb login

In [ ]:
artifact_id = "mikolaj-garbowski-warsaw-university-of-technology/zzsn-projekt/model-vrz7kbm7-step_40000:v0"

multipliers = {
    "abstractionism": {
        9277: -0.3150964677333832,
        663: -0.5785388350486755,
        5708: -0.0997006744146347,
        9391: -0.11663304269313812,
        5406: -0.4028443992137909,
        7666: -0.44385766983032227,
        7677: -0.4122329354286194,
        4490: -0.35500454902648926,
        3709: -0.48453497886657715,
        835: -0.2735205292701721,
    },
    "artist sketch": {
        5518: -0.7107505202293396,
        4320: -0.6993359327316284,
        4802: -0.8586661219596863,
        6179: -0.7275573015213013,
        3718: -0.9358670115470886,
        6905: -0.48176902532577515,
        5087: -0.585903525352478,
        7055: -0.9008421897888184,
        426: -0.48766735196113586,
        2821: -0.6156529188156128,
    },
    "blossom season": {
        8452: -0.2592393159866333,
        6619: -0.27215901017189026,
        5542: 0.0611380934715271,
        1097: -0.23795735836029053,
        4840: -0.2493765652179718,
        707: -0.19611291587352753,
        890: -0.14522433280944824,
        8600: 0.09378417581319809,
        8010: -0.30719760060310364,
        399: -0.5568557977676392,
    },
    "bricks": {
        4400: 0.15077823400497437,
        4382: -0.09356039017438889,
        4912: 0.10672266036272049,
        8600: -0.01585463248193264,
        6998: -0.2661287784576416,
        1492: -0.06717679649591446,
        4667: 0.07528870552778244,
        6368: 0.2788570523262024,
        6984: -0.01777736097574234,
        9673: -0.06676457822322845,
    },
    "byzantine": {
        6984: 0.1577005535364151,
        778: 0.05815662443637848,
        346: 0.2578330636024475,
        5858: 0.010164301842451096,
        3223: -0.0017870028968900442,
        5806: -0.37349051237106323,
        9212: -0.19979119300842285,
        5660: 0.23284435272216797,
        5902: -0.03576475381851196,
        582: 0.4320983588695526,
    },
    "cartoon": {
        6559: -1.02586829662323,
        2532: -0.8932428359985352,
        9874: -0.9567275643348694,
        7044: -0.841062605381012,
        5518: -0.9184314012527466,
        6203: -0.8949572443962097,
        2778: -0.8383631706237793,
        7055: -0.8107898235321045,
        7214: -0.8432652354240417,
        5126: -0.8369369506835938,
    },
}

In [ ]:
import os

os.chdir("zzsn-projekt")
os.getcwd()

In [ ]:
from models.diffusion import WrappedDiffusion, GenerationParams
import matplotlib.pyplot as plt
import torch

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
diffusion = WrappedDiffusion.from_pretrained(artifact_id, device=device,
 safety_checker=None
                                             )

In [ ]:
styles = list(multipliers.keys())

In [ ]:
def make_with_intervention(diffusion, style, multipliers, gamma=1.0):
    params = GenerationParams(
        prompts=[f"A british shorthair cat in {style} style"],
        num_seeds=1,
        num_inference_steps=50,
        guidance_scale=7.5,
    )
    dictionary_multipliers = {k: gamma * v for k, v in multipliers[style].items()}
    return diffusion.generate_with_intervention_2(
        params, dictionary_multipliers=dictionary_multipliers
    )[0].image


def generate(gamma=1.0):
    with_intervention = [
        make_with_intervention(diffusion, style, multipliers) for style in styles
    ]

    params = GenerationParams(
        prompts=[f"A british shorthair cat in {style} style" for style in styles],
        num_seeds=1,
        num_inference_steps=50,
        guidance_scale=7.5,
    )
    results_no_interventions = diffusion.generate(params)
    imgs_no_intervention = [r.image for r in results_no_interventions]
    return imgs_no_intervention, with_intervention

In [ ]:
def plot(imgs_without, imgs_with):
    num_images = len(styles)

    fig, axes = plt.subplots(2, num_images, figsize=(num_images * 3, 6))

    for i, image in enumerate(imgs_without):
        axes[0, i].imshow(image)
        axes[0, i].set_title(f"{styles[i]}")
        axes[0, i].axis("off")

    for i, image in enumerate(imgs_with):
        axes[1, i].imshow(image)
        axes[1, i].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
r1 = generate(1.0)

In [ ]:
plot(r1[0], r1[1])

In [ ]:
r2 = generate(-1.0)

In [ ]:
plot(r2[0], r2[1])

In [ ]:
r3 = generate(100)

In [ ]:
plot(r3[0], r3[1])